In [21]:
import Gmsh: gmsh
import Pkg; Pkg.add("Distributions")
using Gridap, GridapGmsh
using Random
using Distributions

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [22]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square");
    gmsh.model.geo.addPoint(0.0, 0.0, 0.0, h, 1); # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1.0, 0.0, 0.0, h, 2);
    gmsh.model.geo.addPoint(1.0, 1.0, 0.0, h, 3);
    gmsh.model.geo.addPoint(0.0, 1.0, 0.0, h, 4);
    gmsh.model.geo.addLine(1, 2, 1); 
    gmsh.model.geo.addLine(2, 3, 2); # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3, 4, 3);
    gmsh.model.geo.addLine(4, 1, 4);
    
    gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1);

    gmsh.model.geo.addPlaneSurface([1], 1);

    gmsh.model.geo.synchronize();

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")

    gmsh.model.mesh.generate(2); # dimension is 2
    
    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end


create_square_model (generic function with 1 method)

In [23]:
h = 0.03; # target mesh size
model = create_square_model(h) # fix above function above using the tutorial
order = 1
reffe = ReferenceFE(lagrangian, Float64, order)
V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000166709s, CPU 0.000166s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0207863s, CPU 0.02077s)
Info    : 1441 nodes 2884 elements


TrialFESpace()

In [24]:
Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

GenericMeasure()

In [25]:
n=7
Δt, T = 2.0^(-n), 2*π # time step, final time
num_steps = Int(round(T / Δt))

804

In [26]:
f(x) = x[1]*(1-x[1]) * x[2]*(1-x[2])
g = x->f(x) 

#19 (generic function with 1 method)

In [27]:
u₀ = x -> 0.0
w₀ = x -> 0.0

#23 (generic function with 1 method)

In [28]:
m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

M = assemble_matrix(m, U, V)
K = assemble_matrix(k, U, V)

# LHS
A = (1/Δt^2)*M + (1/4)*K

1305×1305 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8861 stored entries:
⎡⡑⣬⡁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣺⡾⠬⢿⣷⣶⠆⠈⣩⣛⣆⢢⠨⠄⡘⢗⣿⣷⢞⎤
⎢⠁⠈⠻⣦⣀⠀⡀⠀⠈⠎⠐⡆⠀⡈⠐⠄⠀⠤⠀⠄⠀⠀⠛⠐⠈⠢⢀⢀⢈⡼⣾⡋⠁⠆⠀⠃⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠘⠿⣧⡅⠀⢑⡥⡀⠁⠀⠂⠈⠀⠀⠦⠀⠀⠀⠀⠀⠀⠭⠀⠀⣮⡊⢷⠪⢅⣦⣽⣣⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠉⣿⣿⣴⠌⣀⠁⠠⠁⠀⠀⠀⠀⠀⠰⠀⠀⠈⠂⠁⢀⢨⠹⣟⣳⠤⢘⣡⣍⣷⠄⠅⠀⠀⠀⎥
⎢⠀⠀⡢⠄⠕⡴⡐⠟⠻⣦⣟⣝⢍⡚⠠⠂⠰⠣⠂⠔⠀⠀⠀⠄⠤⡉⢉⡘⢾⠾⣼⣵⣩⠏⠪⠒⠀⠀⠀⠀⎥
⎢⠀⠀⠰⠤⠄⠈⠄⠘⣟⢽⣿⢟⡷⠸⣯⣮⡴⡿⠀⠀⠀⠀⢄⡇⠀⠈⠝⢾⢋⡱⡼⠼⢿⣹⡘⡩⢐⡀⠀⠄⎥
⎢⠀⠀⡀⠠⠠⠀⠄⠂⣣⠱⣙⡋⣻⣾⡷⢋⢻⡹⣂⠰⣄⠀⠂⢈⠧⡚⠺⠦⡴⠮⠞⣾⣪⡵⣍⣽⡔⠀⢁⡁⎥
⎢⠀⠀⠐⠄⠂⠀⠀⠀⠠⠂⡫⣿⡽⢋⡻⣮⣿⣊⣚⠷⣬⠀⠷⠂⠧⡠⣘⠄⠯⠗⠬⢎⣵⣶⡺⢽⠀⠁⠁⠁⎥
⎢⠀⠀⠀⡄⠠⡄⠀⠀⠴⡂⣴⡯⣟⡲⡻⢻⢵⣷⢯⡷⡚⢉⣛⡷⡻⣞⣰⣾⢼⣽⢞⣟⣵⣷⣡⣜⠩⣉⠈⠉⎥
⎢⠀⠀⠀⠄⠀⠀⢀⡀⢈⠄⠀⠀⢈⡘⢾⡜⢯⡷⠱⢆⣩⡶⢧⡦⣌⣥⢺⡦⣽⣼⡍⣙⣧⣿⣻⡍⡹⠲⠖⠀⎥
⎢⣠⣠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠂⠛⡞⢈⢣⡾⠻⣦⣔⡐⡉⡁⣼⠁⠐⣠⠴⢺⠋⢿⡿⠛⢲⣼⣧⣱⎥
⎢⡚⡏⢛⠀⠀⠀⠢⠀⠀⠄⠤⠵⡈⢀⠹⠃⢿⡼⠩⡷⢐⠹⢿⢗⣾⡻⢜⠻⠸⠼⢋⠛⢣⠤⠂⡏⠳⢟⣽⡥⎥
⎢⢿⣷⠢⡀⠃⠃⠁⢀⡄⠣⡀⠀⣩⠣⠉⡣⣻⢮⠆⣽⠇⠨⣾⡻⢛⢔⡰⠁⡔⠑⣉⡧⢽⡽⠛⣺⣦⡪⢍⢵⎥
⎢⠸⠟⠀⢐⡠⣤⣆⡒⣃⠰⣳⣅⠺⡆⠒⠜⣰⣾⠺⡶⠖⠛⣶⡑⠔⠊⢱⣶⢔⢀⢴⢔⣄⣶⢤⢢⢊⢛⡾⠏⎥
⎢⡆⣠⣂⡴⢮⣌⢿⣹⣺⡗⢏⡰⡰⡏⢯⠇⣖⣷⣓⣿⠐⣠⣒⡆⢔⠉⠐⢑⣵⣿⣷⢾⢧⢟⣵⠌⠠⣒⣄⢁⎥
⎢⠻⢼⡾⠻⠎⢆⣀⢃⢖⣿⣒⡏⣺⣥⡢⢇⣾⢵⣇⢩⣰⣃⣯⠐⠧⡼⢐⢗⣹⣟⢿⢗⡷⣽⡛⢮⠀⡸⠪⡂⎥
⎢⡈⡒⠡⠄⣌⣿⡅⢾⡧⠞⣟⣳⢎⡾⢱⣿⢵⣿⣭⣿⣯⣄⠉⡖⣗⡷⢠⣽⣭⢗⣝⣯⣿⣿⣿⠻⠠⣫⠁⠈⎥
⎢⣀⠡⠤⠀⠉⠚⠙⠟⢪⠂⡖⡨⣇⣽⣞⣎⣁⢾⡟⠾⣿⠋⡬⠤⣻⣠⠠⣓⡑⠟⡻⣌⣿⡛⣿⣿⣁⠈⠛⠀⎥
⎢⣽⣵⠀⠀⠀⠀⠁⠁⠀⠀⠐⠰⠐⠉⠄⠀⡇⢢⢳⡊⣘⣶⣽⢆⡨⡻⣮⢐⢠⢢⣀⡠⡤⣢⡁⠘⢿⣷⠲⣶⎥
⎣⣹⢟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠄⠅⠰⠅⠀⡆⠀⠘⠁⢍⣻⠗⡿⢇⣕⡾⠏⠄⢙⠪⠢⡁⠀⠛⠀⢸⣦⣵⢟⎦

In [29]:
uh0 = interpolate_everywhere(u₀, U)   # U^0
uh1 = interpolate_everywhere(x -> u₀(x) + Δt * w₀(x), U)   # U^1

SingleFieldFEFunction():
 num_cells: 2744
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 3035142464380748857

In [30]:
if !isdir("tmp") mkdir("tmp") end

createpvd("wave") do pvd
    pvd[0.0] = createvtk(Ω, "tmp/stochasticwavecorrect_0.vtu",
                         cellfields=["uh" => uh0])
    pvd[Δt]  = createvtk(Ω, "tmp/stochasticwavecorrect_1.vtu",
                         cellfields=["uh" => uh1])

    # dof-vectors
    uvec_prev = get_free_dof_values(uh0)  # U^{n-1}
    uvec_curr = get_free_dof_values(uh1)  # U^{n}

    for n in 1:(num_steps-1)
        t_n = n*Δt
        Δβ = rand(Normal(0,sqrt(Δt)))
        b(v) = ∫(g*v)dΩ
        Fn = assemble_vector(b, V)
        noise =  Δβ * Fn

        # RHS
        rhs = ((2/Δt^2)*M - (1/2)*K) * uvec_curr +
              (-(1/Δt^2)*M -(1/4)*K) * uvec_prev + noise/Δt

        # Solve for next step
        uvec_next = A \ rhs
        uh_next = FEFunction(U, uvec_next) # solve for next value

        t_next = (n+1)*Δt
        vtufilename = "tmp/stochasticwavecorrect_$(n+1).vtu"
        pvd[t_next] = createvtk(Ω, vtufilename, cellfields=["uh" => uh_next])

        #update
        uvec_prev = uvec_curr
        uvec_curr = uvec_next
    end
end

806-element Vector{String}:
 "wave.pvd"
 "tmp/stochasticwavecorrect_0.vtu"
 "tmp/stochasticwavecorrect_1.vtu"
 "tmp/stochasticwavecorrect_2.vtu"
 "tmp/stochasticwavecorrect_3.vtu"
 "tmp/stochasticwavecorrect_4.vtu"
 "tmp/stochasticwavecorrect_5.vtu"
 "tmp/stochasticwavecorrect_6.vtu"
 "tmp/stochasticwavecorrect_7.vtu"
 "tmp/stochasticwavecorrect_8.vtu"
 "tmp/stochasticwavecorrect_9.vtu"
 "tmp/stochasticwavecorrect_10.vtu"
 "tmp/stochasticwavecorrect_11.vtu"
 ⋮
 "tmp/stochasticwavecorrect_793.vtu"
 "tmp/stochasticwavecorrect_794.vtu"
 "tmp/stochasticwavecorrect_795.vtu"
 "tmp/stochasticwavecorrect_796.vtu"
 "tmp/stochasticwavecorrect_797.vtu"
 "tmp/stochasticwavecorrect_798.vtu"
 "tmp/stochasticwavecorrect_799.vtu"
 "tmp/stochasticwavecorrect_800.vtu"
 "tmp/stochasticwavecorrect_801.vtu"
 "tmp/stochasticwavecorrect_802.vtu"
 "tmp/stochasticwavecorrect_803.vtu"
 "tmp/stochasticwavecorrect_804.vtu"